In [25]:
import os
import pandas as pd
import numpy as np
import pyodbc
import warnings

warnings.filterwarnings('ignore')

# Carregamento dos dados

### Tabela Rides

In [89]:
df_rides= pd.read_csv("rides.csv")
df_rides.head()

,ride_id,user_id,start_station_id,end_station_id,start_time,end_time,distance_km
0,1,542,17,5,2024-03-25 10:06:00,2024-03-25 10:24:55.074889,3.78
1,2,804,2,23,2024-02-05 16:53:00,2024-02-05 17:01:49.215435,1.76
2,3,328,13,23,2024-08-27 04:46:00,2024-08-27 05:00:10.059262,2.83
3,4,52,22,2,2024-01-03 00:39:00,2024-01-03 01:17:10.690811,7.64
4,5,449,23,10,2024-01-29 17:22:00,2024-01-29 18:28:32.193009,13.31


In [90]:
df_rides.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ride_id           15000 non-null  int64  
 1   user_id           15000 non-null  int64  
 2   start_station_id  15000 non-null  int64  
 3   end_station_id    15000 non-null  int64  
 4   start_time        15000 non-null  object 
 5   end_time          15000 non-null  object 
 6   distance_km       15000 non-null  float64
dtypes: float64(1), int64(4), object(2)
memory usage: 820.4+ KB


In [91]:
df_rides.duplicated().sum()

np.int64(0)

In [92]:
# converter colunas de data e hora

df_rides['start_time'] = pd.to_datetime(df_rides['start_time'])
df_rides['end_time'] = pd.to_datetime(df_rides['end_time'])
df_rides.dtypes

ride_id                      int64
user_id                      int64
start_station_id             int64
end_station_id               int64
start_time          datetime64[ns]
end_time            datetime64[ns]
distance_km                float64
dtype: object

In [128]:
df_rides.to_csv('f_rides.csv', index=False)

In [94]:
df_stations= pd.read_csv("stations.csv")
df_stations.head()

,station_id,station_name,capacity,lat,lon
0,1,Megan Manors St,10,46.728847,43.737302
1,2,Ryan Islands St,30,-86.964204,20.624754
2,3,Fisher Stravenue St,20,-66.690303,25.872797
3,4,Jimenez Summit St,20,72.592864,-167.735942
4,5,Taylor Fall St,10,35.634175,2.486946


In [95]:
df_stations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   station_id    25 non-null     int64  
 1   station_name  25 non-null     object 
 2   capacity      25 non-null     int64  
 3   lat           25 non-null     float64
 4   lon           25 non-null     float64
dtypes: float64(2), int64(2), object(1)
memory usage: 1.1+ KB


In [96]:
df_stations.duplicated().sum()

np.int64(0)

In [129]:
df_stations.to_csv('dim_stations.csv', index=False)

In [98]:
df_users= pd.read_csv("users.csv")
df_users.head()

,user_id,username,age,membership_level,created_at
0,1,enewman,33,Casual,2024-03-25
1,2,oliverdaniel,28,Casual,2025-01-20
2,3,pmartinez,35,Casual,2024-11-20
3,4,joseph98,42,Casual,2024-11-08
4,5,xstrong,28,Casual,2024-03-07


In [99]:
df_users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   user_id           1000 non-null   int64 
 1   username          1000 non-null   object
 2   age               1000 non-null   int64 
 3   membership_level  1000 non-null   object
 4   created_at        1000 non-null   object
dtypes: int64(2), object(3)
memory usage: 39.2+ KB


In [100]:
df_users.duplicated().sum()

np.int64(0)

In [101]:
#conversão da coluna de created_at

df_users["created_at"] = pd.to_datetime(df_users['created_at'])
df_users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   user_id           1000 non-null   int64         
 1   username          1000 non-null   object        
 2   age               1000 non-null   int64         
 3   membership_level  1000 non-null   object        
 4   created_at        1000 non-null   datetime64[ns]
dtypes: datetime64[ns](1), int64(2), object(2)
memory usage: 39.2+ KB


In [130]:
df_users.to_csv('dim_users.csv', index=False)

# Inserção dos dados no banco SQL

### Conexão com o SQL Server

In [2]:
dados_conexao = (
    "Driver={SQL Server};" 
    "Server=DESKTOP-JOTJ1VE;"
    "Trusted_Connection=yes"
)

conexao = pyodbc.connect(dados_conexao)

try:
    conexao = pyodbc.connect(dados_conexao)
    print('Conexão bem-sucedida')
except pyodbc.Error as e:
    print('Erro na conexão:', e)

Conexão bem-sucedida


### Criação do banco de dados

In [132]:
# criar o banco de dados

basedados = "Stationbikes"

# Função para criar o banco de dados
def criar_banco(dados_conexao, basedados):
    try:
        conexao = pyodbc.connect(dados_conexao, autocommit=True)
        cursor = conexao.cursor()
        cursor.execute(f"CREATE DATABASE {basedados}")
        print(f"Banco de dados {basedados} criado.") 
        conexao.close()
    except pyodbc.Error as e:
        print(f"Erro ao criar banco de dados: {e}")

criar_banco(dados_conexao, basedados)

Banco de dados Stationbikes criado.


### Inserção das tabelas

In [133]:
dados_conexao = (
    "Driver={SQL Server};" 
    "Server=DESKTOP-JOTJ1VE;"
    "database=Stationbikes;"
    "Trusted_Connection=yes"
)

conexao = pyodbc.connect(dados_conexao)

try:
    conexao = pyodbc.connect(dados_conexao)
    print('Conexão bem-sucedida')
except pyodbc.Error as e:
    print('Erro na conexão:', e)

csv_dir = r'C:\Users\Patrícia\Desktop\Estudos dados\base de dados para projetos\Data Analysis Project with SQL'

tabelas =["f_rides.csv", "dim_stations.csv", "dim_users.csv"]

def criar_tabelas_inserir_dados(tabelas):
    csv_path = os.path.join(csv_dir, tabelas)
    df = pd.read_csv(csv_path)
  
    nome_tabela = os.path.splitext(tabelas)[0]
    
    criar_tabela_sql = f"CREATE TABLE [{nome_tabela}] ("
    for col_nome, col_tipo in zip(df.columns, df.dtypes):
        tipo_str = str(col_tipo)
        
        if 'int' in tipo_str:
            criar_tabela_sql += f"[{col_nome}] INT, "
        elif 'float' in tipo_str:
            criar_tabela_sql += f"[{col_nome}] FLOAT, "
        elif 'datetime' in tipo_str:
            criar_tabela_sql += f"[{col_nome}] DATETIME,"
        else:
            criar_tabela_sql += f"[{col_nome}] NVARCHAR(MAX), "
    criar_tabela_sql = criar_tabela_sql.rstrip(', ') + ")"
    
    cursor = conexao.cursor()
    cursor.execute(f"IF OBJECT_ID('{nome_tabela}', 'U') IS NOT NULL DROP TABLE {nome_tabela}")
    cursor.execute(criar_tabela_sql)
    conexao.commit()
    
    for index, linha in df.iterrows():
        inserir_sql = f"INSERT INTO {nome_tabela} VALUES ("
        for valor in linha:
            if pd.isnull(valor):
                inserir_sql += "NULL, "
            else:
                inserir_sql += f"'{valor}', "
        inserir_sql = inserir_sql.rstrip(', ') + ")"
        cursor.execute(inserir_sql)
    conexao.commit()

for tabela in tabelas:
    criar_tabelas_inserir_dados(tabela)

#conexao.close()

print("Tabelas criadas no SQL Server.")


Conexão bem-sucedida
Tabelas criadas no SQL Server.


In [14]:
conexao.close()

In [3]:
server = 'DESKTOP-JOTJ1VE'
database = 'Stationbikes'

conexao = pyodbc.connect('DRIVER={SQL Server};SERVER='+server+';DATABASE='+database+';Trusted_Connection=yes;')

### Queries

In [4]:
query_teste = """
    SELECT TOP 5* FROM f_rides
"""


Qteste = pd.read_sql(query_teste, conexao)
print(Qteste)

   ride_id  user_id  start_station_id  end_station_id           start_time  \
0        1      542                17               5  2024-03-25 10:06:00   
1        2      804                 2              23  2024-02-05 16:53:00   
2        3      328                13              23  2024-08-27 04:46:00   
3        4       52                22               2  2024-01-03 00:39:00   
4        5      449                23              10  2024-01-29 17:22:00   

                     end_time  distance_km  
0  2024-03-25 10:24:55.074889         3.78  
1  2024-02-05 17:01:49.215435         1.76  
2  2024-08-27 05:00:10.059262         2.83  
3  2024-01-03 01:17:10.690811         7.64  
4  2024-01-29 18:28:32.193009        13.31  


1. Existe algum usuário cadastrado na tabela users que nunca realizou uma viagem? 

In [4]:
query_1 = """
    SELECT user_id FROM dim_users
    WHERE user_id NOT IN (SELECT user_id FROM f_rides);
"""

Q1 = pd.read_sql(query_1, conexao)
print(Q1)

Empty DataFrame
Columns: [user_id]
Index: []


Todos os usuários cadastrados no sistema possuem pelo menos uma viagem realizada.

2. Padrão de Uso (Horário): Em qual hora do dia ocorre o maior volume de viagens? 

In [5]:
query_2 = """
    WITH Periodos_Corridas AS(
        SELECT
        	ride_id,
        	CASE
        		WHEN CAST(start_time as time) >= '05:00:00' AND CAST(start_time as time) < '12:00:00' THEN 'Manhã'
        		WHEN CAST(start_time as time) < '18:00:00' THEN 'Tarde'
        		ELSE 'Noite'
        	END AS Periodo
        FROM f_rides)
    SELECT
    	Periodo,
    	COUNT(ride_id) AS Total_corridas
    FROM Periodos_Corridas
    GROUP BY Periodo
    ORDER BY Total_corridas DESC;
"""

Q2 = pd.read_sql(query_2, conexao)
print(Q2)

  Periodo  Total_corridas
0   Tarde            7687
1   Manhã            5259
2   Noite            2054


3. Rotas Populares: Quais são os 5 pares de estações (origem e destino) que mais aparecem nas viagens? Exiba o nome das estações, não apenas o ID.

In [6]:
query_3 = """
     SELECT TOP 5
        s_start.station_name AS estacao_origem,
        s_end.station_name AS estacao_destino,
        COUNT(*) AS quantidade_viagens
    FROM f_rides r
    JOIN dim_stations s_start ON r.start_station_id = s_start.station_id
    JOIN dim_stations s_end ON r.end_station_id = s_end.station_id
    GROUP BY s_start.station_name, s_end.station_name
    ORDER BY quantidade_viagens DESC;
"""

Q3 = pd.read_sql(query_3, conexao)
print(Q3)

         estacao_origem    estacao_destino  quantidade_viagens
0      Jennifer Land St  Jimenez Summit St                  42
1  Christine Orchard St  Espinoza Pines St                  38
2   Samuel Extension St  Jimenez Summit St                  38
3        Harper Well St   Bell Villages St                  37
4     Michael Shores St     Brown Shoal St                  37


4. Análise de Retenção (Mês a Mês)
Identifique quais usuários foram "retidos". Um usuário é considerado retido se ele fez pelo menos uma viagem em um mês e voltou a fazer pelo menos uma viagem no mês seguinte.

In [5]:
query_4 = """
    WITH atividade_mensal AS (SELECT DISTINCT
    DATEFROMPARTS(YEAR(end_time), MONTH(end_time), 1) AS ride_month,
    user_id
FROM f_rides)

SELECT 
  pma.ride_month, -- mês atual/referência
  COUNT(DISTINCT pma.user_id) AS previous_users, -- número de clientes no mês anterior
  COUNT(DISTINCT cma.user_id) AS retained_users, -- número de clientes retidos no mês seguinte
  ROUND(COUNT(DISTINCT cma.user_id) * 100.0 / COUNT(DISTINCT pma.user_id), 0) AS retention_rate
FROM atividade_mensal pma -- tabela da esquerda 
LEFT JOIN atividade_mensal AS cma -- utilizando um self join para cruzamento dos dados
  ON DATEADD(month, 1, pma.ride_month) = cma.ride_month -- soma 1 mês ao dado da tabela esquerda pma e encontra o resultado na tabela da direita cma
  AND pma.user_id = cma.user_id 
GROUP BY pma.ride_month
ORDER BY pma.ride_month;
"""

Q4 = pd.read_sql(query_4, conexao)
print(Q4)

    ride_month  previous_users  retained_users  retention_rate
0   2024-01-01             728             498            68.0
1   2024-02-01             691             454            66.0
2   2024-03-01             664             460            69.0
3   2024-04-01             711             491            69.0
4   2024-05-01             702             487            69.0
5   2024-06-01             700             502            72.0
6   2024-07-01             710             488            69.0
7   2024-08-01             691             492            71.0
8   2024-09-01             723             520            72.0
9   2024-10-01             733             525            72.0
10  2024-11-01             717             499            70.0
11  2024-12-01             702               1             0.0
12  2025-01-01               1               0             0.0


5. Tempo de Inatividade entre Viagens (Window Functions)
O Desafio: Para cada usuário, calcule a média de dias que ele passa "parado" entre uma viagem e outra.

In [7]:
query_5 = """
    WITH diff_days_rides AS (SELECT
        user_id,
        DATEDIFF(DAY, LAG(end_time) OVER(PARTITION BY user_id ORDER BY start_time), start_time) AS diff_rides
    FROM f_rides) -- cálculo da diferenças de dias entre a primeira e última viagem para cada usuário

    SELECT
        user_id,
        AVG(diff_rides) AS avg_days_inactive
    FROM diff_days_rides
    GROUP BY user_id
    ORDER BY user_id;
"""

Q5 = pd.read_sql(query_5, conexao)
print(Q5)

     user_id  avg_days_inactive
0          1                 20
1          2                 19
2          3                 21
3          4                 32
4          5                 27
..       ...                ...
995      996                 17
996      997                 21
997      998                 28
998      999                 24
999     1000                 22

[1000 rows x 2 columns]


5.1. Qual a idade e membership level dos usários mais inativos?

In [9]:
query_5_1 = """
WITH diff_days_rides AS (SELECT
    user_id,
    DATEDIFF(DAY, LAG(end_time) OVER(PARTITION BY user_id ORDER BY start_time), start_time) AS diff_rides
FROM f_rides) -- cálculo da diferenças de dias entre a primeira e última viagem para cada usuário

SELECT
    u.membership_level,
    AVG(d.diff_rides) AS avg_days_inactive
FROM diff_days_rides d
LEFT JOIN dim_users u
ON d.user_id = u.user_id
GROUP BY u.membership_level
ORDER BY u.membership_level;
"""
Q5_1= pd.read_sql(query_5_1, conexao)
print(Q5_1)

  membership_level  avg_days_inactive
0           Casual                 22
1       Subscriber                 22


In [13]:
query_5_2 = """
WITH diff_days_rides AS (SELECT
    user_id,
    DATEDIFF(DAY, LAG(end_time) OVER(PARTITION BY user_id ORDER BY start_time), start_time) AS diff_rides
FROM f_rides), -- cálculo da diferenças de dias entre a primeira e última viagem para cada usuário

UsuariosComFaixa AS (
    SELECT 
        user_id,
        age,
        CASE 
            WHEN age < 18 THEN 'Adolescente'
            WHEN age >=18 AND age <=30 THEN 'Jovem adulto'
            WHEN age >30 AND age <=60 THEN 'Adulto'
            ELSE 'Idoso'
        END AS Faixa
    FROM dim_users
)

SELECT
    u.Faixa AS Faixa_Etaria,
    AVG(d.diff_rides) AS avg_days_inactive
FROM diff_days_rides d
LEFT JOIN UsuariosComFaixa u
ON d.user_id = u.user_id
GROUP BY u.Faixa
ORDER BY avg_days_inactive DESC;
"""

Q5_2 = pd.read_sql(query_5_2, conexao)
print(Q5_2)

   Faixa_Etaria  avg_days_inactive
0        Adulto                 23
1  Jovem adulto                 22


6. Análise de "Power Users" 
O Desafio: Crie um relatório que mostre o top 1% dos usuários que mais percorreram distância total. Para esses usuários, calcule:

A distância total.

Uma coluna chamada perfil_uso: Se a média de distância por viagem dele for > 8km, marque como "Explorador", caso contrário, "Comuter" (trabalho/estudo).

In [14]:
query_6 = """
    SELECT 
        TOP 1 PERCENT WITH TIES user_id, -- garante o retorno de dados mesmo com empate
        SUM(distance_km) AS distancia_total,
        CASE
            WHEN AVG(distance_km) > 8 THEN 'Explorador'
            ELSE 'Comuter (trabalho/estudos)'
        END AS Perfil
    FROM f_rides
    GROUP BY user_id
    ORDER BY distancia_total DESC;
"""

Q6 = pd.read_sql(query_6, conexao)
print(Q6)

   user_id  distancia_total                      Perfil
0       43           228.05  Comuter (trabalho/estudos)
1      979           202.95                  Explorador
2      642           189.97  Comuter (trabalho/estudos)
3      752           186.42  Comuter (trabalho/estudos)
4      515           183.71  Comuter (trabalho/estudos)
5      735           183.48  Comuter (trabalho/estudos)
6      883           179.54  Comuter (trabalho/estudos)
7      613           178.55  Comuter (trabalho/estudos)
8      122           176.92  Comuter (trabalho/estudos)
9      258           176.30                  Explorador


7. Horários de pico por Estação
Identifique os horários com maior total de corridas por estação.

In [16]:
query_7 = """
   SELECT
        s.station_name,
        s.capacity,
    DATEPART(HOUR, r.start_time) AS Hora,
    COUNT(r.ride_id) AS total_corridas
    FROM f_rides r
    INNER JOIN dim_stations s
    ON r.start_station_id = s.station_id
    GROUP BY s.station_id, s.station_name,s.capacity, DATEPART(HOUR, r.start_time)
    ORDER BY total_corridas DESC;
"""

Q7 = pd.read_sql(query_7, conexao)
print(Q7)

           station_name  capacity  Hora  total_corridas
0       King Harbors St        30    15              81
1       King Harbors St        30    16              77
2      Edwards Drive St        50    15              77
3       Megan Manors St        10    15              76
4    Krystal Village St        20    15              75
..                  ...       ...   ...             ...
593   Espinoza Pines St        10     1               1
594   Espinoza Pines St        10     3               1
595    Jennifer Land St        20     3               1
596    Jennifer Land St        20    23               1
597         Amy Park St        10    23               1

[598 rows x 4 columns]


8. Análise de Retenção e Cohort
Desafio: Calcule o "tempo de vida" do usuário no sistema: a diferença em dias entre a data de criação da conta (created_at) e a data da sua viagem mais recente.

In [20]:
query_8_1 = """
SELECT 
	u.user_id,
	DATEDIFF(DAY,u.created_at, MAX(r.start_time)) AS Tempo_ativo
FROM dim_users u JOIN f_rides r ON u.user_id = r.user_id
GROUP BY 
    u.user_id, 
    u.created_at;
"""

Q8_1 = pd.read_sql(query_8_1, conexao)
print(Q8_1)

     user_id  Tempo_ativo
0        383          331
1        469          328
2        608          309
3        739          317
4         15          321
..       ...          ...
995        2          -25
996      199          -80
997      325          -39
998      558          -21
999      586          -82

[1000 rows x 2 columns]


Durante a análise de tempo de atividade foram encontrados 973 usuários cuja data da última viagem é inferior a data dde criação da conta, indicando que há um problema na ingestão dos dados dos usuários listados abaixo. 

In [21]:
query_8_2 = """
WITH Tempo_Atividade AS (SELECT 
	u.user_id,
	DATEDIFF(DAY,u.created_at, MAX(r.start_time)) AS Tempo_ativo
FROM dim_users u JOIN f_rides r ON u.user_id = r.user_id
WHERE r.start_time > u.created_at
GROUP BY 
    u.user_id, 
    u.created_at)

SELECT
    AVG(Tempo_ativo) AS Media_Tempo_Ativo
FROM Tempo_Atividade;
"""

Q8_2 = pd.read_sql(query_8_2, conexao)
print(Q8_2)

   Media_Tempo_Ativo
0                160


In [24]:
query_8_3= """
SELECT 
    COUNT(DISTINCT u.user_id) AS total_usuarios_com_erro
FROM dim_users u 
JOIN f_rides r ON u.user_id = r.user_id
WHERE r.start_time < u.created_at;
"""
Q8_3 = pd.read_sql(query_8_3, conexao)
print(Q8_3)

   total_usuarios_com_erro
0                      973


In [23]:
query_8_4 = """
SELECT 
    u.user_id,
    u.username AS usuarios_erro_cadastro
FROM dim_users u JOIN f_rides r ON u.user_id = r.user_id
WHERE r.start_time < u.created_at;
"""

Q8_4 = pd.read_sql(query_8_4, conexao)
print(Q8_4)

      user_id usuarios_erro_cadastro
0         754               dustin75
1           4               joseph98
2         154           hamptoncasey
3         440               ugregory
4         957             erikmeyers
...       ...                    ...
8303      551            miguelsmith
8304      996           kruegersally
8305      729              vschwartz
8306      440               ugregory
8307      967         laurenguerrero

[8308 rows x 2 columns]


## Insights e recomendações:

**1. Engajamento e ciclo de vida do usuário:**

* A análise indica que  todos os usuários cadastrados na plataforma possuem pelo menos uma viagem realizada, ou seja,  o processo de onboarding do aplicativo é altamente eficaz em converter novos cadastros em usuários ativos de forma imediata.
  
* A taxa de retenção mensal (MoM) ao longo de 2024 manteve-se estável entre 66% e 72%, indicando que o produto está maduro no mercado e possui forte aderência do público, ou seja, a maior parte da base de usuários enxerga valor contínuo no serviço.

* Contudo, a análise de inatividade revelou que os usuários passam, em média, entre 10 e 30 dias inativos entre uma corrida e outra. Além disso, não há variação significativa nesse valor ao analisar em nível de faixa etária ou tipo de assinatura, em que há em média 22 dias de inatividade. A partir desse insight recomenda-se criar uma estratégia de recuperação de engajamento via email ou notificação push para alertar o usuário quando atingir 15 dias sem usar o serviço, oferecendo alguns incentivos para reduzir essa janela de tempo.
  
**2. Eficiência Operacional e Logística:**

* Os usuários tendem a usar mais o serviço no período vespertino, que concentra 7687 corridas, com um pico operacional entre 15 e 16 horas. O segundo período com maior pico ocorre entre 6 horas e 7 horas. Esses horários indicam que os clientes provavelmente usam o serviço de bicicletas para ida e volta de trabalho, escola/faculdade.

* Com base nos dados de capacidade e total de corridas por período, recomenda-se expandir a capacidade das principais estações nos horários de pico para evitar falta de bicicletas disponíveis para os clientes.

**3.Perfil dos Power Users (O Top 1%):**

* A maioria dos Top usuários do serviço são do tipo Commuters, ou seja, que fazem em média 8km por viagem, comprovando que os clientes que mais recorrentes da plataforma utilizam as biciletas como condução diária e não para lazer.
* Além disso, o tempo de atividade média dos usuários é de 160 dias, demonstrando que há um ciclo de relacionamento de aproximadamente 5 meses.
* Com base nesses inisghts, recomeda-se criar planos de assinatura focados em benefícios de mobilidade urbana diária para os Power users, como, por exemplo, descontos corporativos a fim de aumentar o tempo de retenção e satisfação dos clientes. 

**4. Governança e Qualidade de Dados:**

Como mencionado anteriormente, foi encontrada uma anomalia na ingestão dos dados de 973 usuários, em que a data de encerramento da viagem é inferior à data de criação da conta na plataforma. Portanto, é necessário reportar o caso ao time de engenharia de dados para solicitar auditação das tabelas de origem, investigando qual falha de sincronização de fusos horários ou outro tipo de erro possa estar ocasionando essa anomalia. 